**Part 4 — GenAI: turn call-centre notes into structured reasons, then recommend**

Task 4.1

In [ ]:

!pip install -q google-genai pydantic

import json
from google import genai
from google.genai import types

MODEL_NAME = "gemini-3.5-flash-lite"


API_KEY = input("Paste your Gemini API key: ")

client = genai.Client(api_key=API_KEY)

print("Gemini client initialized successfully!")


In [ ]:
SCHEMA = {
    "type": "object",

    "properties": {

        "root_cause": {
            "type": "string"
        },

        "confidence": {
            "type": "number"
        },

        "summary": {
            "type": "string"
        },

        "recommended_action": {
            "type": "string"
        }
    },

    "required": [
        "root_cause",
        "confidence",
        "summary",
        "recommended_action"
    ]
}

In [ ]:
SYSTEM_TEXT = """
You are a professional clinic operations analyst.

Your responsibility is to analyse clinic appointment notes,
identify the most likely root cause of a problem,
and recommend a practical action.

Follow the user's instructions carefully.
Do not invent information.
Return only valid JSON matching the supplied schema.
"""

In [ ]:
ROOT_CAUSE_OPTIONS = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
]

def build_prompt(note_text):

    prompt = f"""
CONTEXT:
You are analysing clinic appointment notes.
The clinic wants to understand why an appointment
may have resulted in a problem or a no-show.

ROLE:
Act as an experienced clinic operations analyst.

ACTION:
Read the given note carefully.

Identify the most likely root cause.

Choose root_cause ONLY from the following allowed values:
{ROOT_CAUSE_OPTIONS}

Estimate your confidence between 0 and 1.

Write a short summary of the note in no more than 25 words.

Recommend one practical action that the clinic can take.

Do not invent information that is not present in the note.

TONE:
Professional, factual, clear and concise.

FORMAT:
Return a JSON object containing exactly these four fields:

1. root_cause: string
2. confidence: number between 0 and 1
3. summary: string, maximum 25 words
4. recommended_action: string

Do not return Markdown or explanations outside the JSON.

CLINIC NOTE:
{note_text}
"""

    return prompt

In [ ]:
def analyse_note(note_text):

    prompt = build_prompt(note_text)


    response = client.models.generate_content(

        model=MODEL_NAME,

        contents=prompt,

        config=types.GenerateContentConfig(

            system_instruction=SYSTEM_TEXT,

            temperature=0,

            max_output_tokens=1500,

            response_mime_type="application/json",


            response_schema=SCHEMA,


            #thinking_config=types.ThinkingConfig(
                #thinking_level="minimal"
            #)
        )
    )


    data = json.loads(response.text)


    return data

In [ ]:
notes_df = pd.read_csv("followup_notes.csv")

print(notes_df.head())



  note_id appointment_id                                          note_text
0    N001          A0872  Patient said she never got any reminder messag...
1    N002          A0897  Lives far from the clinic and could not find a...
2    N003          A0526  Patient felt the consultation fee was too high...
3    N004          A0926                                                NaN
4    N005          A0534  Had high fever and body ache on the day, staye...


In [ ]:
print(notes_df.columns.tolist())

['note_id', 'appointment_id', 'note_text']


In [ ]:
note_n001 = notes_df.loc[
    notes_df["note_id"] == "N001",
    "note_text"
].iloc[0]

print("Note N001:")
print(note_n001)

Note N001:
Patient said she never got any reminder message and completely forgot about the visit. Wants to rebook for next week.


In [ ]:
notes_df = pd.read_csv("followup_notes.csv")

note_n001 = notes_df.loc[
    notes_df["note_id"] == "N001",
    "note_text"
].iloc[0]


result = analyse_note(note_n001)

# Print parsed Python dictionary
print(result)
print(type(result))


{'root_cause': 'forgot_or_no_reminder', 'confidence': 1.0, 'summary': 'Patient stated she never received a reminder message, forgot the appointment, and requested to rebook for next week.', 'recommended_action': 'Verify and update patient contact information in the system to ensure future appointment reminders are successfully delivered.'}
<class 'dict'>


Task 4.2

In [ ]:
import time

from pydantic import BaseModel, Field, ValidationError

In [ ]:
class FollowupInsight(BaseModel):

    root_cause: str = Field(min_length=3)

    confidence: float = Field(ge=0, le=1)

    summary: str = Field(min_length=5)

    recommended_action: str = Field(min_length=5)

In [ ]:
def validate_root_cause(root_cause):

    if root_cause not in ROOT_CAUSE_OPTIONS:
        raise ValueError(
            f"Invalid root_cause: {root_cause}"
        )

    return True

In [ ]:
def analyse_note(note_text, return_raw=False):

    # Build CRAFT prompt using the function from Task 4.1
    prompt = build_prompt(note_text)

    # Send request to Gemini
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1500,
            response_mime_type="application/json",
            response_schema=SCHEMA,

        )
    )


    raw_text = response.text or ""


    if return_raw:
        return raw_text


    data = json.loads(raw_text)

    return data

In [ ]:
notes_df = pd.read_csv("followup_notes.csv")

print(notes_df.head())

print("\nColumns:")
print(notes_df.columns.tolist())

  note_id appointment_id                                          note_text
0    N001          A0872  Patient said she never got any reminder messag...
1    N002          A0897  Lives far from the clinic and could not find a...
2    N003          A0526  Patient felt the consultation fee was too high...
3    N004          A0926                                                NaN
4    N005          A0534  Had high fever and body ache on the day, staye...

Columns:
['note_id', 'appointment_id', 'note_text']


In [ ]:

results = []


api_calls = 0



for index, row in notes_df.head(10).iterrows():

    print(f"\nProcessing row {index + 1}...")


    note_id = row["note_id"]
    appointment_id = row["appointment_id"]
    note_text = row["note_text"]




    # Check NaN FIRST because NaN is a float
    if pd.isna(note_text):

        is_blank = True

    else:

        # Then check empty string or spaces
        is_blank = str(note_text).strip() == ""


    if is_blank:

        print("Blank note found. Skipping Gemini API call.")

        record = {
            "note_id": note_id,
            "appointment_id": appointment_id,
            "root_cause": "other",
            "confidence": 0.0,
            "summary": "No note recorded",
            "recommended_action": "Re-call patient",
            "validation_status": "skipped_blank",
            "raw_response": ""
        }

        results.append(record)

        continue




    raw_response = ""
    data = {}
    validation_status = "invalid"

    try:

        # Pause 13 seconds between API calls
        if api_calls > 0:
            time.sleep(13)

        api_calls += 1

        raw_response = analyse_note(
            str(note_text).strip(),
            return_raw=True
        )


        data = json.loads(raw_response)

        if not isinstance(data, dict):
            raise ValueError("Response is not a JSON object")

        # Validate fields using Pydantic
        insight = FollowupInsight(**data)

        # Check root cause against allowed categories
        validate_root_cause(insight.root_cause)

        # All validation checks passed
        validation_status = "valid"

        print("Validation successful!")

    except Exception as e:

        validation_status = "invalid"

        print("Validation failed:", str(e))




    record = {
        "note_id": note_id,
        "appointment_id": appointment_id,

        "root_cause": data.get("root_cause", ""),

        "confidence": data.get("confidence", ""),

        "summary": data.get("summary", ""),

        "recommended_action": data.get(
            "recommended_action", ""
        ),

        "validation_status": validation_status,


        "raw_response": raw_response
    }

    results.append(record)


    print("Status:", validation_status)


print("\nProcessing completed!")
print("Total records processed:", len(results))
print("Total API calls attempted:", api_calls)



Processing row 1...
Validation successful!
Status: valid

Processing row 2...
Validation failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Status: invalid

Processing row 3...
Validation successful!
Status: valid

Processing row 4...
Blank note found. Skipping Gemini API call.

Processing row 5...
Validation successful!
Status: valid

Processing row 6...
Validation failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Status: invalid

Processing row 7...
Validation failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Status: invalid

Process

In [ ]:

output_columns = [
    "note_id",
    "appointment_id",
    "root_cause",
    "confidence",
    "summary",
    "recommended_action",
    "validation_status",
    "raw_response"
]

# Ensure correct column order
results_df = results_df[output_columns]

# Save CSV
results_df.to_csv(
    "notes_analysed.csv",
    index=False
)

print("File saved successfully: notes_analysed.csv")

File saved successfully: notes_analysed.csv


In [ ]:
# Read saved CSV again
final_df = pd.read_csv("notes_analysed.csv")

print(final_df.head(10))

print("\nValidation status counts:")
print(final_df["validation_status"].value_counts())

print("\nTotal records:", len(final_df))

  note_id appointment_id             root_cause  confidence  \
0    N001          A0872  forgot_or_no_reminder        1.00   
1    N002          A0897                    NaN         NaN   
2    N003          A0526            cost_or_fee        1.00   
3    N004          A0926                  other        0.00   
4    N005          A0534   illness_or_emergency        0.98   
5    N006          A0507                    NaN         NaN   
6    N007          A0333                    NaN         NaN   
7    N008          A0222  distance_or_transport        0.95   
8    N009          A0977  forgot_or_no_reminder        0.60   
9    N010          A0596       work_or_schedule        0.95   

                                             summary  \
0  Patient missed the appointment because she nev...   
1                                                NaN   
2  Patient skipped the appointment due to high co...   
3                                   No note recorded   
4  Patient developed a hig

task 4.3

In [ ]:
# Bad reply 1: confidence is greater than 1
bad_reply_1 = {
    "root_cause": "transport",
    "confidence": 1.7,
    "summary": "Patient could not arrange transport.",
    "recommended_action": "Offer a new appointment."
}

# Bad reply 2: recommended_action is missing
bad_reply_2 = {
    "root_cause": "cost",
    "confidence": 0.8,
    "summary": "Patient could not afford the visit."
}


print("TEST 1: Invalid confidence")

try:
    result = FollowupInsight(**bad_reply_1)
    print("Valid:", result)

except ValidationError as e:
    print(e)


print("\nTEST 2: Missing recommended_action")

try:
    result = FollowupInsight(**bad_reply_2)
    print("Valid:", result)

except ValidationError as e:
    print(e)

TEST 1: Invalid confidence
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal

TEST 2: Missing recommended_action
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'cost', 'c... not afford the visit.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing


In [ ]:

from google.genai import types



notes_df = pd.read_csv("followup_notes.csv")


test_notes = notes_df[
    notes_df["note_id"].isin(["N007", "N009"])
]


# Count API calls made in Task 4.3
api_calls_task43 = 0


def classify_with_prompt(note_text, prompt):
    global api_calls_task43

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1500,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(
                thinking_level="minimal"
            )
        )
    )

    api_calls_task43 += 1

    raw_response = response.text
    data = json.loads(raw_response)

    return data, raw_response



def check_allowed_root_cause(data):
    root_cause = data.get("root_cause", "")

    try:
        FollowupInsight(**data)

        if root_cause not in ROOT_CAUSE_OPTIONS:
            return "invalid - not in allowed list"

        return "valid"

    except ValidationError:
        return "invalid - Pydantic error"


results = []

for _, row in test_notes.iterrows():

    note_id = row["note_id"]
    note_text = str(row["note_text"]).strip()

    # V1: Simple prompt
    v1_prompt = f"Classify this note:\n{note_text}"

    # V2: Existing CRAFT prompt
    v2_prompt = build_prompt(note_text)

    # Call Gemini with V1
    try:
        v1_data, v1_raw = classify_with_prompt(
            note_text,
            v1_prompt
        )

        v1_status = check_allowed_root_cause(v1_data)

        results.append({
            "note_id": note_id,
            "prompt": "V1",
            "root_cause": v1_data.get("root_cause", ""),
            "confidence": v1_data.get("confidence", ""),
            "validation": v1_status
        })

    except Exception as e:
        print(f"V1 error for {note_id}: {e}")

        results.append({
            "note_id": note_id,
            "prompt": "V1",
            "root_cause": "",
            "confidence": "",
            "validation": "API/JSON error"
        })


    # Call Gemini with V2
    try:
        v2_data, v2_raw = classify_with_prompt(
            note_text,
            v2_prompt
        )

        v2_status = check_allowed_root_cause(v2_data)

        results.append({
            "note_id": note_id,
            "prompt": "V2",
            "root_cause": v2_data.get("root_cause", ""),
            "confidence": v2_data.get("confidence", ""),
            "validation": v2_status
        })

    except Exception as e:
        print(f"V2 error for {note_id}: {e}")

        results.append({
            "note_id": note_id,
            "prompt": "V2",
            "root_cause": "",
            "confidence": "",
            "validation": "API/JSON error"
        })



comparison_df = pd.DataFrame(results)

print(comparison_df.to_string(index=False))

print("\nAPI calls used in Task 4.3:", api_calls_task43)

note_id prompt                                                        root_cause  confidence                    validation
   N007     V1 Patient forgot the appointment and did not receive a reminder SMS        0.95 invalid - not in allowed list
   N007     V2                                             forgot_or_no_reminder        0.95                         valid
   N009     V1               Patient unreachable due to phone being switched off        0.95 invalid - not in allowed list
   N009     V2                                             forgot_or_no_reminder        0.60                         valid

API calls used in Task 4.3: 4


Task 4.4

In [ ]:
notes_df = pd.read_csv("notes_analysed.csv")
appointments_df = pd.read_csv("appointments_clean.csv")

print("Notes shape:", notes_df.shape)
print("Appointments shape:", appointments_df.shape)

print(notes_df.head())
print(appointments_df.head())

Notes shape: (10, 8)
Appointments shape: (990, 16)
  note_id appointment_id             root_cause  confidence  \
0    N001          A0872  forgot_or_no_reminder        1.00   
1    N002          A0897                    NaN         NaN   
2    N003          A0526            cost_or_fee        1.00   
3    N004          A0926                  other        0.00   
4    N005          A0534   illness_or_emergency        0.98   

                                             summary  \
0  Patient missed the appointment because she nev...   
1                                                NaN   
2  Patient skipped the appointment due to high co...   
3                                   No note recorded   
4  Patient developed a high fever and body ache o...   

                                  recommended_action validation_status  \
0  Verify and update the patient's contact inform...             valid   
1                                                NaN           invalid   
2  Provide 

In [ ]:
# Replace missing root causes with "unknown"
merged_df["root_cause"] = (
    merged_df["root_cause"].fillna("unknown")
)

# Replace missing reminder values with "unknown"
merged_df["reminder_sent"] = (
    merged_df["reminder_sent"].fillna("unknown")
)

cross_tab = pd.crosstab(
    merged_df["root_cause"],
    merged_df["reminder_sent"],
    margins=True
)

print("Root Cause vs Reminder Sent")
print(cross_tab)

Root Cause vs Reminder Sent
reminder_sent          No  Yes  All
root_cause                         
cost_or_fee             0    1    1
distance_or_transport   0    1    1
forgot_or_no_reminder   1    1    2
illness_or_emergency    0    1    1
other                   1    0    1
unknown                 2    1    3
work_or_schedule        1    0    1
All                     5    5   10
